In [1]:
# IMPORTS AND CONFIG
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from itertools import product
import json
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

PARQUET_PATH     = r"C:\Users\super\Desktop\Data Thesis\final_df_clustered.parquet"
MAX_SEQ_LEN      = 80
BATCH_SIZE       = 32
EPOCHS           = 50
PATIENCE         = 10
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_RANDOM_CONFIGS = 25   # random configs tried for LSTM and Transformer

print(f"Using device: {DEVICE}")

Using device: cpu


In [2]:
# LOAD DATA AND DEFINE FEATURES
print("Loading data...")
final_df = pd.read_parquet(PARQUET_PATH)

gaze_cols = [c for c in final_df.columns if "gaze" in c]
pose_cols = [c for c in final_df.columns if "pose" in c]
au_r_cols = [c for c in final_df.columns if c.startswith("AU") and c.endswith("_r")]
au_c_cols = [c for c in final_df.columns if c.startswith("AU") and c.endswith("_c")]

# model input features, behavioral_cluster excluded
FEATURE_COLS = au_r_cols + au_c_cols + gaze_cols + pose_cols
N_FEATURES   = len(FEATURE_COLS)

assert "behavioral_cluster" in final_df.columns, \
    "behavioral_cluster missing — run Grouping.ipynb first"
assert "behavioral_cluster" not in FEATURE_COLS, \
    "behavioral_cluster must not be used as a model feature"

print(f"  Timesteps:    {len(final_df):,}")
print(f"  Clips:        {final_df['clip_id'].nunique():,}")
print(f"  Participants: {final_df['participant_id'].nunique()}")
print(f"  Clusters:     {final_df['behavioral_cluster'].nunique()}")
print(f"  Features:     {N_FEATURES}")
print(f"  Class balance (engaged): {final_df['engagement'].mean():.2%}")

Loading data...
  Timesteps:    798,300
  Clips:        12,093
  Participants: 102
  Clusters:     2
  Features:     49
  Class balance (engaged): 80.55%


In [3]:
# PARTICIPANT LEVEL TRAIN/VAL/TEST SPLIT
participant_label = (
    final_df.groupby("participant_id")["engagement"]
    .agg(lambda x: int(x.mean() >= 0.5))
)

engaged_participants    = participant_label[participant_label == 1].index.values.copy()
disengaged_participants = participant_label[participant_label == 0].index.values.copy()

rng = np.random.default_rng(SEED)
rng.shuffle(engaged_participants)
rng.shuffle(disengaged_participants)

def stratified_split(arr, train_frac=0.70, val_frac=0.15):
    n       = len(arr)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    return arr[:n_train], arr[n_train:n_train + n_val], arr[n_train + n_val:]

train_eng, val_eng, test_eng = stratified_split(engaged_participants)
train_dis, val_dis, test_dis = stratified_split(disengaged_participants)

train_participants = np.concatenate([train_eng, train_dis])
val_participants   = np.concatenate([val_eng,   val_dis])
test_participants  = np.concatenate([test_eng,  test_dis])

train_mask = final_df["participant_id"].isin(train_participants)
val_mask   = final_df["participant_id"].isin(val_participants)
test_mask  = final_df["participant_id"].isin(test_participants)

# clip-level metadata, behavioral_cluster carried for evaluation only
clip_meta = (
    final_df.groupby("clip_id")
    .agg(
        engagement         = ("engagement",         "first"),
        participant_id     = ("participant_id",     "first"),
        behavioral_cluster = ("behavioral_cluster", "first")
    )
    .reset_index()
)

train_clips = clip_meta[clip_meta["participant_id"].isin(train_participants)].reset_index(drop=True)
val_clips   = clip_meta[clip_meta["participant_id"].isin(val_participants)].reset_index(drop=True)
test_clips  = clip_meta[clip_meta["participant_id"].isin(test_participants)].reset_index(drop=True)

print(f"Split — train: {train_mask.sum():,} timesteps ({len(train_participants)} participants)")
print(f"      — val:   {val_mask.sum():,} timesteps ({len(val_participants)} participants)")
print(f"      — test:  {test_mask.sum():,} timesteps ({len(test_participants)} participants)")
print(f"\nClips — train: {len(train_clips):,}  val: {len(val_clips):,}  test: {len(test_clips):,}")
print(f"\nCluster distribution in test set:")
print(test_clips["behavioral_cluster"].value_counts().sort_index())

Split — train: 558,893 timesteps (71 participants)
      — val:   118,123 timesteps (14 participants)
      — test:  121,284 timesteps (17 participants)

Clips — train: 8,484  val: 1,832  test: 1,777

Cluster distribution in test set:
behavioral_cluster
0     516
1    1261
Name: count, dtype: int64


In [4]:
# FEATURE STANDARDIZATION
# fit scaler on training timesteps only
scaler = StandardScaler()
scaler.fit(final_df.loc[train_mask, FEATURE_COLS])
final_df[FEATURE_COLS] = scaler.transform(final_df[FEATURE_COLS])

n_train_pos   = int(train_clips["engagement"].sum())
n_train_neg   = len(train_clips) - n_train_pos
class_weights = torch.tensor(
    [n_train_pos / n_train_neg, 1.0], dtype=torch.float32
).to(DEVICE)

print(f"Class weights — disengaged: {class_weights[0]:.4f}  engaged: {class_weights[1]:.4f}")

Class weights — disengaged: 4.2338  engaged: 1.0000


In [5]:
# EVALUATION HELPERS
def compute_metrics(y_true, y_pred, split_name=""):
    acc      = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro",  zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average="micro",  zero_division=0)
    print(f"\n── {split_name} ───────────────────────────────────────")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}   (primary metric)")
    print(f"  Micro F1:  {micro_f1:.4f}")
    print(classification_report(y_true, y_pred,
                                target_names=["Disengaged", "Engaged"],
                                zero_division=0))
    return {"accuracy": acc, "macro_f1": macro_f1, "micro_f1": micro_f1}


def fairness_metrics(y_true, y_pred, cluster_ids, split_name="", n_bootstrap=1000):
    MEANINGFUL_THRESHOLD = 0.10
    cluster_ids          = np.array(cluster_ids)
    unique_clusters      = sorted(np.unique(cluster_ids))

    def per_cluster_stats(yt, yp, cids):
        rows = []
        for cid in unique_clusters:
            mask = cids == cid
            if mask.sum() == 0:
                continue
            yt_c, yp_c = yt[mask], yp[mask]
            n  = len(yt_c)
            tp = int(((yp_c == 1) & (yt_c == 1)).sum())
            tn = int(((yp_c == 0) & (yt_c == 0)).sum())
            fp = int(((yp_c == 1) & (yt_c == 0)).sum())
            fn = int(((yp_c == 0) & (yt_c == 1)).sum())
            rows.append({
                "cluster":        cid,
                "n_clips":        n,
                "tpr":            tp / (tp + fn) if (tp + fn) > 0 else np.nan,
                "fpr":            fp / (fp + tn) if (fp + tn) > 0 else np.nan,
                "fnr":            fn / (tp + fn) if (tp + fn) > 0 else np.nan,
                "error_rate":     (fp + fn) / n,
                "low_confidence": n < 5
            })
        return pd.DataFrame(rows)

    def disparity_measures(stats_df):
        return {
            "eod":               stats_df["tpr"].max()        - stats_df["tpr"].min(),
            "fpr_gap":           stats_df["fpr"].max()        - stats_df["fpr"].min(),
            "fnr_gap":           stats_df["fnr"].max()        - stats_df["fnr"].min(),
            "worst_group_error": stats_df["error_rate"].max()
        }

    df       = per_cluster_stats(y_true, y_pred, cluster_ids)
    reliable = df[~df["low_confidence"]]
    point    = disparity_measures(reliable)

    # Bootstrap CIs
    boot_stats = {"eod": [], "fpr_gap": [], "fnr_gap": [], "worst_group_error": []}
    rng_boot   = np.random.default_rng(SEED)
    n          = len(y_true)
    for _ in range(n_bootstrap):
        idx      = rng_boot.integers(0, n, size=n)
        boot_df  = per_cluster_stats(y_true[idx], y_pred[idx], cluster_ids[idx])
        boot_rel = boot_df[~boot_df["low_confidence"]]
        if len(boot_rel["cluster"].unique()) < 2:
            continue
        b = disparity_measures(boot_rel)
        for key in boot_stats:
            boot_stats[key].append(b[key])

    ci = {
        key: (np.percentile(vals, 2.5), np.percentile(vals, 97.5))
        for key, vals in boot_stats.items()
    }

    print(f"\n── Fairness Metrics: {split_name} ──────────────────────────")
    for label, key in [
        ("Equal Opportunity Diff (EOD)", "eod"),
        ("FPR gap                      ", "fpr_gap"),
        ("FNR gap                      ", "fnr_gap"),
        ("Worst-group error             ", "worst_group_error"),
    ]:
        flag = "  ⚠ MEANINGFUL" if point[key] > MEANINGFUL_THRESHOLD else "  ✓"
        print(f"  {label}: {point[key]:.4f}  "
              f"95% CI [{ci[key][0]:.4f}, {ci[key][1]:.4f}]{flag}")

    if df["low_confidence"].any():
        print(f"  ⚠ Low-confidence clusters (<5 clips): "
              f"{df.loc[df['low_confidence'], 'cluster'].tolist()}")

    print(f"\n  Per-cluster breakdown:")
    print(df.round(4).to_string(index=False))

    return df, {**point, "ci": ci}

In [6]:
# SEQUENCE DATASET AND DATALOADERS
class EngagementDataset(Dataset):
    def __init__(self, clips_df, full_df, max_len=MAX_SEQ_LEN):
        self.max_len  = max_len
        self.clip_ids = clips_df["clip_id"].tolist()
        self.labels   = clips_df["engagement"].tolist()
        self.clusters = clips_df["behavioral_cluster"].tolist()  # evaluation only
        relevant      = full_df[full_df["clip_id"].isin(self.clip_ids)]
        self.clip_dict = {
            cid: grp.sort_values("timestep")[FEATURE_COLS].values.astype(np.float32)
            for cid, grp in relevant.groupby("clip_id")
        }

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, idx):
        seq = self.clip_dict.get(
            self.clip_ids[idx],
            np.zeros((1, N_FEATURES), dtype=np.float32)
        )
        T = seq.shape[0]
        if T >= self.max_len:
            seq = seq[:self.max_len]
        else:
            seq = np.vstack([seq,
                             np.zeros((self.max_len - T, N_FEATURES),
                                      dtype=np.float32)])
        return torch.tensor(seq), torch.tensor(self.labels[idx], dtype=torch.long)


print("Building sequence datasets...")
train_dataset = EngagementDataset(train_clips, final_df)
val_dataset   = EngagementDataset(val_clips,   final_df)
test_dataset  = EngagementDataset(test_clips,  final_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
print("Done.")

Building sequence datasets...
Done.


In [7]:
# SHARED SEQUENCE MODEL AND TRAINING HELPERS
def train_and_validate(model, learning_rate):
    criterion         = nn.CrossEntropyLoss(weight=class_weights)
    optimiser         = torch.optim.Adam(model.parameters(), lr=learning_rate)
    best_val_f1       = -1.0
    best_state        = None
    epochs_no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimiser.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            train_loss += loss.item()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                preds.extend(
                    model(X_batch.to(DEVICE)).argmax(dim=1).cpu().numpy()
                )
                labels.extend(y_batch.numpy())

        val_f1 = f1_score(labels, preds, average="macro", zero_division=0)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | "
                  f"train loss: {train_loss/len(train_loader):.4f} | "
                  f"val macro F1: {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1       = val_f1
            best_state        = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"  Early stopping at epoch {epoch} "
                      f"(best val macro F1: {best_val_f1:.4f})")
                break

    model.load_state_dict(best_state)
    return best_val_f1, model


def evaluate_sequence_model(model, loader, clips_df, split_name, model_name):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds.extend(
                model(X_batch.to(DEVICE)).argmax(dim=1).cpu().numpy()
            )
            labels.extend(y_batch.numpy())

    y_true      = np.array(labels)
    y_pred      = np.array(preds)
    cluster_ids = clips_df["behavioral_cluster"].values.astype(int)

    metrics             = compute_metrics(y_true, y_pred,
                              split_name=f"{model_name} — {split_name}")
    fairness_df, disparity = fairness_metrics(y_true, y_pred, cluster_ids,
                              split_name=model_name)
    return metrics, fairness_df, disparity

In [8]:
# MODEL ARCHITECTURE DEFINITIONS
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe           = torch.zeros(max_len, d_model)
        position     = torch.arange(0, max_len).unsqueeze(1).float()
        div_term     = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2,
                 dropout=0.1, bidirectional=True, n_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = dropout if num_layers > 1 else 0.0
        )
        out_dim         = hidden_dim * 2 if bidirectional else hidden_dim
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(out_dim, n_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.classifier(self.dropout(out.mean(dim=1)))


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1, n_classes=2,
                 max_len=MAX_SEQ_LEN):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=max_len,
                                             dropout=dropout)
        encoder_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.encoder    = nn.TransformerEncoder(encoder_layer,
                                                num_layers=num_layers)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = self.pos_enc(self.input_proj(x))
        x = self.encoder(x)
        return self.classifier(self.dropout(x.mean(dim=1)))

In [9]:
# LOGISTIC REGRESSION: GRID SEARCH + EVALUATION
print("=" * 60)
print("MODEL 1: LOGISTIC REGRESSION — GRID SEARCH")
print("=" * 60)

def build_pooled_features(clips_df, df):
    rows    = []
    grouped = df[df["clip_id"].isin(clips_df["clip_id"])].groupby("clip_id")
    for _, row in clips_df.iterrows():
        cid = row["clip_id"]
        if cid not in grouped.groups:
            continue
        g = grouped.get_group(cid)[FEATURE_COLS]
        rows.append(np.concatenate([
            g.mean().values,
            g.std().fillna(0).values,
            [row["engagement"], row["behavioral_cluster"]]
        ]))
    cols = ([f"{c}_mean" for c in FEATURE_COLS] +
            [f"{c}_std"  for c in FEATURE_COLS] +
            ["engagement", "behavioral_cluster"])
    return pd.DataFrame(rows, columns=cols)

print("Building pooled features...")
train_pool = build_pooled_features(train_clips, final_df)
val_pool   = build_pooled_features(val_clips,   final_df)
test_pool  = build_pooled_features(test_clips,  final_df)

pool_feat_cols = [c for c in train_pool.columns
                  if c not in ("engagement", "behavioral_cluster")]

X_train_lr = train_pool[pool_feat_cols].values.astype(np.float32)
y_train_lr = train_pool["engagement"].values.astype(int)
X_val_lr   = val_pool[pool_feat_cols].values.astype(np.float32)
y_val_lr   = val_pool["engagement"].values.astype(int)
X_test_lr  = test_pool[pool_feat_cols].values.astype(np.float32)
y_test_lr  = test_pool["engagement"].values.astype(int)
clusters_test_lr = test_pool["behavioral_cluster"].values.astype(int)

# majority-class baseline
majority_pred     = np.ones_like(y_test_lr)
majority_acc      = accuracy_score(y_test_lr, majority_pred)
majority_macro_f1 = f1_score(y_test_lr, majority_pred, average="macro", zero_division=0)
majority_micro_f1 = f1_score(y_test_lr, majority_pred, average="micro", zero_division=0)
print(f"\nMajority-class baseline — accuracy: {majority_acc:.4f}  macro F1: {majority_macro_f1:.4f}")

# grid search
LR_GRID = {
    "C":            [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    "penalty":      ["l2", "l1"],
    "class_weight": ["balanced", None],
}
lr_combos = [dict(zip(LR_GRID.keys(), v)) for v in product(*LR_GRID.values())]
print(f"\nSearching {len(lr_combos)} LR configurations...")

lr_results = []
for i, params in enumerate(lr_combos):
    solver = "liblinear" if params["penalty"] == "l1" else "lbfgs"
    try:
        m = LogisticRegression(
            C=params["C"], penalty=params["penalty"],
            class_weight=params["class_weight"],
            solver=solver, max_iter=1000, random_state=SEED
        )
        m.fit(X_train_lr, y_train_lr)
        val_f1 = f1_score(y_val_lr, m.predict(X_val_lr),
                          average="macro", zero_division=0)
        lr_results.append({**params, "solver": solver, "val_macro_f1": val_f1})
        if (i + 1) % 8 == 0:
            print(f"  [{i+1}/{len(lr_combos)}] best so far: "
                  f"{max(r['val_macro_f1'] for r in lr_results):.4f}")
    except Exception as e:
        print(f"  Config {i} failed: {e}")

lr_results_df = pd.DataFrame(lr_results).sort_values("val_macro_f1", ascending=False)
print("\nTop 5 LR configurations:")
print(lr_results_df.head(5).to_string(index=False))

lr_best = lr_results_df.iloc[0].to_dict()
print(f"\n✓ Best LR config: {lr_best}")

# retrain best config and evaluate
best_lr_model = LogisticRegression(
    C=lr_best["C"], penalty=lr_best["penalty"],
    class_weight=lr_best["class_weight"],
    solver=lr_best["solver"], max_iter=1000, random_state=SEED
)
best_lr_model.fit(X_train_lr, y_train_lr)
lr_metrics = compute_metrics(y_test_lr, best_lr_model.predict(X_test_lr),
                             split_name="LR — Test")
_, lr_disparity = fairness_metrics(
    y_test_lr, best_lr_model.predict(X_test_lr),
    clusters_test_lr, split_name="LR"
)

with open("lr_best_params.json", "w") as f:
    json.dump({k: v for k, v in lr_best.items() if k != "val_macro_f1"}, f, indent=2)
print("Saved: lr_best_params.json")

MODEL 1: LOGISTIC REGRESSION — GRID SEARCH
Building pooled features...

Majority-class baseline — accuracy: 0.7997  macro F1: 0.4443

Searching 24 LR configurations...
  [8/24] best so far: 0.6094
  [16/24] best so far: 0.6425
  [24/24] best so far: 0.6484

Top 5 LR configurations:
    C penalty class_weight    solver  val_macro_f1
100.0      l1         None liblinear      0.648384
 10.0      l1         None liblinear      0.648384
 10.0      l2         None     lbfgs      0.647857
100.0      l2         None     lbfgs      0.644441
  1.0      l1         None liblinear      0.642531

✓ Best LR config: {'C': 100.0, 'penalty': 'l1', 'class_weight': None, 'solver': 'liblinear', 'val_macro_f1': 0.6483839480582152}

── LR — Test ───────────────────────────────────────
  Accuracy:  0.7991
  Macro F1:  0.5450   (primary metric)
  Micro F1:  0.7991
              precision    recall  f1-score   support

  Disengaged       0.49      0.13      0.20       356
     Engaged       0.82      0.97      

In [10]:
# LSTM: RANDOM SEARCH + EVALUATION
print("=" * 60)
print("MODEL 2: LSTM — RANDOM SEARCH")
print("=" * 60)

LSTM_PARAM_SPACE = {
    "hidden_dim":    [32, 64, 128],
    "num_layers":    [1, 2, 3],
    "dropout":       [0.0, 0.1, 0.2, 0.3],
    "learning_rate": [1e-4, 5e-4, 1e-3, 5e-3],
    "bidirectional": [True, False],
}

all_lstm_combos = list(product(*LSTM_PARAM_SPACE.values()))
rng_search      = np.random.default_rng(SEED + 1)
rng_search.shuffle(all_lstm_combos)
lstm_combos = all_lstm_combos[:N_RANDOM_CONFIGS]

print(f"Searching {N_RANDOM_CONFIGS} of {len(all_lstm_combos)} possible LSTM configs...\n")

lstm_results = []
for i, vals in enumerate(lstm_combos):
    params = dict(zip(LSTM_PARAM_SPACE.keys(), vals))
    print(f"[{i+1}/{N_RANDOM_CONFIGS}] {params}")
    model = LSTMClassifier(
        input_dim     = N_FEATURES,
        hidden_dim    = params["hidden_dim"],
        num_layers    = params["num_layers"],
        dropout       = params["dropout"],
        bidirectional = params["bidirectional"]
    ).to(DEVICE)
    val_f1, _ = train_and_validate(model, params["learning_rate"])
    print(f"  → val macro F1: {val_f1:.4f}\n")
    lstm_results.append({**params, "val_macro_f1": val_f1})

lstm_results_df = pd.DataFrame(lstm_results).sort_values("val_macro_f1", ascending=False)
print("Top 5 LSTM configurations:")
print(lstm_results_df.head(5).to_string(index=False))

lstm_best = lstm_results_df.iloc[0].to_dict()
print(f"\n✓ Best LSTM config: {lstm_best}")

# retrain best config from scratch and evaluate
print("\nRetraining best LSTM from scratch...")
best_lstm_model = LSTMClassifier(
    input_dim     = N_FEATURES,
    hidden_dim    = int(lstm_best["hidden_dim"]),
    num_layers    = int(lstm_best["num_layers"]),
    dropout       = lstm_best["dropout"],
    bidirectional = lstm_best["bidirectional"]
).to(DEVICE)
_, best_lstm_model = train_and_validate(best_lstm_model, lstm_best["learning_rate"])

lstm_metrics, _, lstm_disparity = evaluate_sequence_model(
    best_lstm_model, test_loader, test_clips, "Test", "LSTM"
)

torch.save(best_lstm_model.state_dict(), "lstm_best.pt")
with open("lstm_best_params.json", "w") as f:
    json.dump({k: v for k, v in lstm_best.items() if k != "val_macro_f1"}, f, indent=2)
print("Saved: lstm_best.pt, lstm_best_params.json")

MODEL 2: LSTM — RANDOM SEARCH
Searching 25 of 288 possible LSTM configs...

[1/25] {'hidden_dim': 128, 'num_layers': 2, 'dropout': 0.1, 'learning_rate': 0.001, 'bidirectional': False}
  Epoch   1 | train loss: 0.5725 | val macro F1: 0.6615
  Epoch  10 | train loss: 0.2671 | val macro F1: 0.5766
  Early stopping at epoch 11 (best val macro F1: 0.6615)
  → val macro F1: 0.6615

[2/25] {'hidden_dim': 32, 'num_layers': 3, 'dropout': 0.0, 'learning_rate': 0.0001, 'bidirectional': False}
  Epoch   1 | train loss: 0.6902 | val macro F1: 0.5093
  Epoch  10 | train loss: 0.4902 | val macro F1: 0.6276
  Epoch  20 | train loss: 0.4394 | val macro F1: 0.6227
  Early stopping at epoch 21 (best val macro F1: 0.6314)
  → val macro F1: 0.6314

[3/25] {'hidden_dim': 32, 'num_layers': 1, 'dropout': 0.3, 'learning_rate': 0.0005, 'bidirectional': True}
  Epoch   1 | train loss: 0.6248 | val macro F1: 0.5936
  Epoch  10 | train loss: 0.3811 | val macro F1: 0.6235
  Early stopping at epoch 19 (best val macr

In [11]:
# TRANSFORMER: RANDOM SEARCH + EVALUATION
print("=" * 60)
print("MODEL 3: TRANSFORMER — RANDOM SEARCH")
print("=" * 60)

# only (d_model, nhead) pairs where d_model is divisible by nhead
TRANSFORMER_PARAM_SPACE = {
    "d_model_nhead":   [(32, 2), (32, 4), (64, 4), (64, 8), (128, 4), (128, 8)],
    "num_layers":      [1, 2, 3],
    "dim_feedforward": [64, 128, 256],
    "dropout":         [0.0, 0.1, 0.2, 0.3],
    "learning_rate":   [1e-4, 5e-4, 1e-3, 5e-3],
}

all_tr_combos = list(product(*TRANSFORMER_PARAM_SPACE.values()))
rng_search.shuffle(all_tr_combos)
tr_combos = all_tr_combos[:N_RANDOM_CONFIGS]

print(f"Searching {N_RANDOM_CONFIGS} of {len(all_tr_combos)} possible Transformer configs...\n")

tr_results = []
for i, vals in enumerate(tr_combos):
    params         = dict(zip(TRANSFORMER_PARAM_SPACE.keys(), vals))
    d_model, nhead = params.pop("d_model_nhead")
    params["d_model"] = d_model
    params["nhead"]   = nhead
    print(f"[{i+1}/{N_RANDOM_CONFIGS}] {params}")
    model = TransformerClassifier(
        input_dim       = N_FEATURES,
        d_model         = d_model,
        nhead           = nhead,
        num_layers      = params["num_layers"],
        dim_feedforward = params["dim_feedforward"],
        dropout         = params["dropout"]
    ).to(DEVICE)
    val_f1, _ = train_and_validate(model, params["learning_rate"])
    print(f"  → val macro F1: {val_f1:.4f}\n")
    tr_results.append({**params, "val_macro_f1": val_f1})

tr_results_df = pd.DataFrame(tr_results).sort_values("val_macro_f1", ascending=False)
print("Top 5 Transformer configurations:")
print(tr_results_df.head(5).to_string(index=False))

tr_best = tr_results_df.iloc[0].to_dict()
print(f"\n✓ Best Transformer config: {tr_best}")

# retrain best config from scratch and evaluate
print("\nRetraining best Transformer from scratch...")
best_tr_model = TransformerClassifier(
    input_dim       = N_FEATURES,
    d_model         = int(tr_best["d_model"]),
    nhead           = int(tr_best["nhead"]),
    num_layers      = int(tr_best["num_layers"]),
    dim_feedforward = int(tr_best["dim_feedforward"]),
    dropout         = tr_best["dropout"]
).to(DEVICE)
_, best_tr_model = train_and_validate(best_tr_model, tr_best["learning_rate"])

tr_metrics, _, tr_disparity = evaluate_sequence_model(
    best_tr_model, test_loader, test_clips, "Test", "Transformer"
)

torch.save(best_tr_model.state_dict(), "transformer_best.pt")
with open("transformer_best_params.json", "w") as f:
    json.dump({k: v for k, v in tr_best.items() if k != "val_macro_f1"}, f, indent=2)
print("Saved: transformer_best.pt, transformer_best_params.json")

MODEL 3: TRANSFORMER — RANDOM SEARCH
Searching 25 of 864 possible Transformer configs...

[1/25] {'num_layers': 2, 'dim_feedforward': 64, 'dropout': 0.1, 'learning_rate': 0.005, 'd_model': 32, 'nhead': 2}
  Epoch   1 | train loss: 0.6164 | val macro F1: 0.6120
  Epoch  10 | train loss: 0.5153 | val macro F1: 0.5637
  Early stopping at epoch 13 (best val macro F1: 0.6272)
  → val macro F1: 0.6272

[2/25] {'num_layers': 3, 'dim_feedforward': 128, 'dropout': 0.1, 'learning_rate': 0.0005, 'd_model': 32, 'nhead': 4}
  Epoch   1 | train loss: 0.6154 | val macro F1: 0.5761
  Epoch  10 | train loss: 0.4538 | val macro F1: 0.5865
  Early stopping at epoch 14 (best val macro F1: 0.6329)
  → val macro F1: 0.6329

[3/25] {'num_layers': 3, 'dim_feedforward': 256, 'dropout': 0.0, 'learning_rate': 0.0005, 'd_model': 128, 'nhead': 8}
  Epoch   1 | train loss: 0.6015 | val macro F1: 0.6581
  Epoch  10 | train loss: 0.1968 | val macro F1: 0.6114
  Early stopping at epoch 11 (best val macro F1: 0.6581)
 

In [12]:
# FINAL SUMMARY
print("=" * 60)
print("FINAL SUMMARY — OVERALL PERFORMANCE (test set)")
print("=" * 60)

overall_summary = pd.DataFrame({
    "Model":    ["Majority Baseline", "Logistic Regression", "LSTM", "Transformer"],
    "Accuracy": [majority_acc,        lr_metrics["accuracy"],
                 lstm_metrics["accuracy"], tr_metrics["accuracy"]],
    "Macro F1": [majority_macro_f1,   lr_metrics["macro_f1"],
                 lstm_metrics["macro_f1"], tr_metrics["macro_f1"]],
    "Micro F1": [majority_micro_f1,   lr_metrics["micro_f1"],
                 lstm_metrics["micro_f1"], tr_metrics["micro_f1"]],
})
print(overall_summary.round(4).to_string(index=False))

print("\n" + "=" * 60)
print("FINAL SUMMARY — FAIRNESS DISPARITIES (test set)")
print("=" * 60)
print("⚠ = disparity exceeds 10 pp (practically meaningful)\n")

MEANINGFUL = 0.10
def fmt(val, ci_lo, ci_hi, threshold=MEANINGFUL):
    flag = " ⚠" if val > threshold else "  "
    return f"{val:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]{flag}"

rows = []
for name, disp in [("LR", lr_disparity), ("LSTM", lstm_disparity),
                   ("Transformer", tr_disparity)]:
    ci = disp["ci"]
    rows.append({
        "Model":             name,
        "EOD":               fmt(disp["eod"],               *ci["eod"]),
        "FPR gap":           fmt(disp["fpr_gap"],           *ci["fpr_gap"]),
        "FNR gap":           fmt(disp["fnr_gap"],           *ci["fnr_gap"]),
        "Worst-group error": fmt(disp["worst_group_error"], *ci["worst_group_error"],
                                 threshold=0.0),
    })
print(pd.DataFrame(rows).to_string(index=False))

print("\n" + "=" * 60)
print("EFFECTIVENESS THRESHOLDS FOR PHASE 3 (Section 3.5.3)")
print("=" * 60)
print("The fair model must beat all three of these transformer baselines:\n")
print(f"  Macro F1 floor:          ≥ {tr_metrics['macro_f1'] - 0.05:.4f}  "
      f"(within 5 pp of {tr_metrics['macro_f1']:.4f})")
print(f"  Worst-group error ceil:  <  {tr_disparity['worst_group_error']:.4f}")
print(f"  EOD ceil:                <  {tr_disparity['eod']:.4f}")

# save full search results
lr_results_df["model"]  = "LR"
lstm_results_df["model"] = "LSTM"
tr_results_df["model"]  = "Transformer"
all_results = pd.concat([lr_results_df, lstm_results_df, tr_results_df],
                        ignore_index=True, sort=False)
all_results.to_csv("hyperparameter_results.csv", index=False)
print("\nSaved: hyperparameter_results.csv")

FINAL SUMMARY — OVERALL PERFORMANCE (test set)
              Model  Accuracy  Macro F1  Micro F1
  Majority Baseline    0.7997    0.4443    0.7997
Logistic Regression    0.7991    0.5450    0.7991
               LSTM    0.7563    0.6185    0.7563
        Transformer    0.7203    0.6142    0.7203

FINAL SUMMARY — FAIRNESS DISPARITIES (test set)
⚠ = disparity exceeds 10 pp (practically meaningful)

      Model                        EOD                    FPR gap                    FNR gap          Worst-group error
         LR 0.0106  [0.0005, 0.0351]   0.1131  [0.0323, 0.2004] ⚠ 0.0106  [0.0005, 0.0351]   0.2209  [0.1903, 0.2578] ⚠
       LSTM 0.1531  [0.1080, 0.2030] ⚠ 0.2400  [0.1404, 0.3396] ⚠ 0.1531  [0.1080, 0.2030] ⚠ 0.3081  [0.2696, 0.3491] ⚠
Transformer 0.1508  [0.0958, 0.2073] ⚠ 0.4581  [0.3584, 0.5507] ⚠ 0.1508  [0.0958, 0.2073] ⚠ 0.3023  [0.2707, 0.3430] ⚠

EFFECTIVENESS THRESHOLDS FOR PHASE 3 (Section 3.5.3)
The fair model must beat all three of these transformer baselines: